# Notebook 03: lax and Control Flow

## Why This Matters

JAX's JIT compilation traces your Python function statically. This means Python `for` loops get **unrolled** into the computation graph -- fine for short loops, catastrophic for sequences of length 1000. JAX provides `jax.lax.scan`, `lax.cond`, and `lax.while_loop` as primitives that compile efficiently without unrolling. Understanding when and why to use these is essential for implementing RNNs, SSMs (like Mamba), or any sequential computation in JAX.


In [ ]:
%pip install -q jax jaxlib numpy matplotlib

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax
import numpy as np
import time
import matplotlib.pyplot as plt
print("Ready.")

## 1. Why Python Loops Unroll in JIT

When JAX traces your function, it executes Python control flow **immediately** using abstract values. A Python `for i in range(100)` loop runs 100 times **during tracing**, producing 100 separate JAX operations in the HLO graph. This means:
- Compilation time scales with loop count
- Binary size grows with loop count
- For long sequences, compilation becomes impractical

The fix: use `lax.scan` -- a JAX-native loop that compiles to a **single fused operation** regardless of loop count.


In [ ]:
# Demonstrating loop unrolling overhead
def sum_python_loop(x):
    total = 0.0
    for i in range(len(x)):
        total = total + x[i]
    return total

x = jnp.ones(100)

# Time compilation for different loop counts
for n in [10, 100, 500]:
    arr = jnp.ones(n)
    
    def loop_fn(x):
        total = 0.0
        for i in range(len(x)):
            total = total + x[i]
        return total
    
    jit_fn = jax.jit(loop_fn)
    t0 = time.perf_counter()
    _ = jit_fn(arr).block_until_ready()
    compile_time = (time.perf_counter() - t0) * 1000
    print(f'Loop count={n:4d}: compile time = {compile_time:.2f} ms')

print('(Notice compile time grows with loop count -- the graph is unrolled)')

## 2. lax.scan: Efficient Sequential Loops

`lax.scan` is JAX's workhorse for sequential computation. It compiles to a single loop primitive in XLA -- no unrolling, constant compile time, memory-efficient.

```python
# Signature:
final_carry, stacked_outputs = lax.scan(fn, init_carry, xs)
```

- `fn(carry, x) -> (new_carry, output)` -- pure function
- `init_carry` -- initial state
- `xs` -- input sequence, batched along first axis
- Returns: `final_carry` (last state) and all outputs stacked

Use case: **anywhere you have a sequential recurrence** -- RNNs, cumulative operations, state-space models.


In [ ]:
# lax.scan basics: cumulative sum
def scan_cumsum(carry, x):
    new_carry = carry + x
    return new_carry, new_carry  # (state, output)

x = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
final, cumsum = lax.scan(scan_cumsum, 0.0, x)
print('Input:       ', x)
print('Cumsum:      ', cumsum)
print('Final carry: ', final)
print('jnp.cumsum:  ', jnp.cumsum(x))

# Compile time comparison: scan vs Python loop
for n in [100, 1000, 5000]:
    arr = jnp.ones(n)
    
    @jax.jit
    def scan_sum(x):
        _, outputs = lax.scan(scan_cumsum, 0.0, x)
        return outputs
    
    t0 = time.perf_counter()
    _ = scan_sum(arr).block_until_ready()
    t = (time.perf_counter() - t0) * 1000
    print(f'scan n={n:5d}: compile+run = {t:.2f} ms (constant compile time)')

## 3. Implementing an RNN with lax.scan

The classic RNN recurrence: $h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$

In JAX, the carry is the hidden state $h_t$. `lax.scan` naturally implements this with no Python loop.


In [ ]:
# Simple Elman RNN with lax.scan
key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)

hidden_size = 32
input_size = 16
seq_len = 100

# Initialize parameters
Wh = jax.random.normal(k1, (hidden_size, hidden_size)) * 0.1
Wx = jax.random.normal(k2, (input_size, hidden_size)) * 0.1
b  = jnp.zeros(hidden_size)

def rnn_step(h, x):
    # h: (hidden_size,), x: (input_size,)
    new_h = jnp.tanh(h @ Wh + x @ Wx + b)
    return new_h, new_h  # carry, output

# Input sequence: (seq_len, input_size)
inputs = jax.random.normal(k3, (seq_len, input_size))

h0 = jnp.zeros(hidden_size)  # initial hidden state

@jax.jit
def run_rnn(inputs, h0):
    h_final, h_all = lax.scan(rnn_step, h0, inputs)
    return h_final, h_all

h_final, h_all = run_rnn(inputs, h0)
print('Input shape:       ', inputs.shape)
print('All hidden states: ', h_all.shape)  # (seq_len, hidden_size)
print('Final hidden state:', h_final.shape)

# Benchmark: scan vs Python loop
def rnn_python_loop(inputs, h0):
    h = h0
    hs = []
    for x in inputs:
        h = jnp.tanh(h @ Wh + x @ Wx + b)
        hs.append(h)
    return h, jnp.stack(hs)

_ = run_rnn(inputs, h0)  # warmup

t0 = time.perf_counter()
for _ in range(100):
    _ = run_rnn(inputs, h0)[0].block_until_ready()
scan_time = (time.perf_counter() - t0) / 100 * 1000

t0 = time.perf_counter()
for _ in range(10):
    _ = rnn_python_loop(inputs, h0)[0].block_until_ready()
loop_time = (time.perf_counter() - t0) / 10 * 1000

print(f'\nlax.scan RNN: {scan_time:.2f} ms/call')
print(f'Python loop:  {loop_time:.2f} ms/call')
print(f'Speedup: {loop_time/scan_time:.1f}x')

## 4. lax.cond: Conditionals Inside JIT

Python `if` on a JAX array fails inside JIT because the value is unknown at trace time. `lax.cond` is the JIT-compatible conditional:

```python
lax.cond(condition, true_fn, false_fn, *operands)
```

**Important**: Both branches are traced (but only one executes at runtime). Side effects in either branch are problematic.


In [ ]:
# lax.cond: conditional inside jit

@jax.jit
def safe_normalize(x):
    norm = jnp.linalg.norm(x)
    # Can't use: if norm < 1e-8: return x
    return lax.cond(
        norm < 1e-8,
        lambda x: x,           # true branch: return as-is
        lambda x: x / norm,    # false branch: normalize
        x
    )

x1 = jnp.array([0.0, 0.0, 0.0])  # near-zero
x2 = jnp.array([3.0, 4.0, 0.0])  # normal

print('Zero vector:', safe_normalize(x1))
print('Normal vec: ', safe_normalize(x2))
print('Norm of normalized:', jnp.linalg.norm(safe_normalize(x2)))

# lax.switch: multiple branches
@jax.jit
def apply_activation(x, mode):
    # mode must be an integer index
    return lax.switch(
        mode,
        [lambda x: jax.nn.relu(x),
         lambda x: jnp.tanh(x),
         lambda x: jax.nn.sigmoid(x)],
        x
    )

x = jnp.array([-1.0, 0.0, 1.0, 2.0])
for i, name in enumerate(['relu', 'tanh', 'sigmoid']):
    print(f'{name}: {apply_activation(x, i)}')

## 5. lax.while_loop: Dynamic Iteration Inside JIT

When the number of iterations is data-dependent (not known at compile time), use `lax.while_loop`. This compiles to a single while loop in XLA without unrolling.

```python
final_state = lax.while_loop(cond_fn, body_fn, init_state)
```

Use case: iterative solvers (Newton's method, conjugate gradient), beam search.


In [ ]:
# lax.while_loop: Newton's method for sqrt
def newton_sqrt(a):
    """Compute sqrt(a) via Newton's method: x_{n+1} = (x_n + a/x_n) / 2"""
    def cond(state):
        x, prev_x = state
        return jnp.abs(x - prev_x) > 1e-6
    
    def body(state):
        x, _ = state
        new_x = (x + a / x) / 2.0
        return (new_x, x)
    
    init = (a / 2.0, 0.0)  # initial guess, prev=0
    final_x, _ = lax.while_loop(cond, body, init)
    return final_x

newton_sqrt_jit = jax.jit(newton_sqrt)

for a in [4.0, 9.0, 2.0, 100.0]:
    result = newton_sqrt_jit(jnp.array(a))
    print(f'sqrt({a:5.1f}): JAX Newton = {result:.6f}, jnp.sqrt = {jnp.sqrt(a):.6f}')

# lax.while_loop for gradient descent
def gd_to_convergence(init_x, learning_rate=0.1, tol=1e-5, max_iter=1000):
    """Minimize f(x) = x^2 using gradient descent until convergence."""
    def cond(state):
        x, i = state
        return (jnp.abs(x) > tol) & (i < max_iter)
    
    def body(state):
        x, i = state
        grad = 2.0 * x  # grad of x^2
        return (x - learning_rate * grad, i + 1)
    
    x_final, n_iters = lax.while_loop(cond, body, (init_x, 0))
    return x_final, n_iters

gd_jit = jax.jit(gd_to_convergence)
x_final, n_iters = gd_jit(jnp.array(10.0))
print(f'\nGD result: x = {x_final:.8f}, iterations = {n_iters}')

## Summary

### Key Concepts

| Primitive | Use when | Compile time | Notes |
|-----------|----------|-------------|-------|
| Python `for` loop | Loop count < ~20, static | Grows linearly | Gets unrolled into HLO graph |
| `lax.scan` | Sequential recurrence, any length | Constant | Best for RNNs, SSMs, cumsum |
| `lax.cond` | Data-dependent binary conditional | Constant | Both branches are traced |
| `lax.switch` | Data-dependent multi-branch | Constant | Index into list of functions |
| `lax.while_loop` | Unknown iteration count | Constant | Good for iterative solvers |

**Key rule**: If the number of iterations depends on array values, you must use a `lax` primitive. If it depends only on Python values (shapes, hyperparameters), Python control flow is fine.

**Next**: `04_flax_nnx_fundamentals.ipynb` -- building neural networks with Flax NNX.
